# Turnover rate rolling average: compare (ticket 2011)

Loads the results of `experiment.ipynb` and picks the single best setup: the lowest worker-weighted aggregate error on test B (hidden last 12 months). There is no tolerance, so the winner is the lowest error. Workplace MAE, bias and jumpiness are shown alongside, as is the mean-of-rates baseline if it was run.

In [ ]:
RESULTS_PREFIX = "s3://skillsforcare-nonprod/sagemaker-notebooks/turnover-rolling-avg/"
RUN_LABEL = "run1"
LOCAL_ROOT = None
REPO_DIR = "repo"

In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path(REPO_DIR).resolve()))

import experiments.rate_rolling_average.experiment_utils as x

LOCAL = LOCAL_ROOT is not None
RESULTS_DIR = (
    str(Path(LOCAL_ROOT) / "results" / RUN_LABEL) if LOCAL else f"{RESULTS_PREFIX.rstrip('/')}/{RUN_LABEL}"
)


def load_all(prefix: str) -> pl.DataFrame:
    names = x.list_results(prefix, RESULTS_DIR, LOCAL)
    return pl.concat([x.read_result(name, RESULTS_DIR, LOCAL) for name in names]) if names else pl.DataFrame()


pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(60)

## Test A

Split, window and fill cap by aggregate error.

In [ ]:
load_all("test_a/").drop("method").sort(x.AGGREGATE_ERROR)

## Test B and baseline

The first row of test B is the setup to put in the pipeline. The `mean_of_rates` row, if present, is the baseline on the winner.

In [ ]:
test_b_df = pl.concat([load_all("test_b/"), load_all("baseline/")], how="vertical_relaxed").sort(x.AGGREGATE_ERROR)
test_b_df

In [ ]:
winner = test_b_df.filter(~pl.col("variant").str.ends_with("_mean_of_rates")).row(0, named=True)
print(
    f"Pipeline constants: ROLLING_AVERAGE_PERIOD = '{winner['window_months']}mo', "
    f"EXTRAPOLATION_PERIOD = '{winner['fill_cap']}', EXTRAPOLATION_METHOD = '{winner['method']}', "
    f"group levels = {x.SPLITS[winner['split']]}"
)